**10 언어 모델을 위한 신경망**
====

**10-3 대규모 언어 모델로 텍스트 생성하기**
----

In [1]:
# EXAONE-3.5 토크나이저 로드

from transformers import AutoTokenizer

exaone_tokenizer = AutoTokenizer.from_pretrained("LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/563 [00:00<?, ?B/s]

In [3]:
# EXAONE-3.5 모델 불러오기

from transformers import pipeline

pipe = pipeline(task="text-generation", 
                model="LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct",
                tokenizer=exaone_tokenizer,
                device=0, trust_remote_code=True)

config.json: 0.00B [00:00, ?B/s]

configuration_exaone.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- configuration_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_exaone.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- modeling_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.65G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Device set to use cuda:0


In [4]:
# 채팅 템플릿 만들기

messages = [
    {"role": "system",
     "content": "너는 쇼핑몰 홈페이지에 올라온 질문에 대답하는 Q&A 챗봇이야. \
                확정적인 답변을 하지 말고 제품 담당자가 정확한 답변을 하기 위해 \
                시간이 필요하다는 간단하고 친절한 답변을 생성해줘."},
    {"role": "user", "content": "이 다이어리에 내년도 공휴일이 표시되어 있나요?"}
]

In [5]:
# 파이프라인 객체를 호출

pipe(messages, max_new_tokens=200)

[{'generated_text': [{'role': 'system',
    'content': '너는 쇼핑몰 홈페이지에 올라온 질문에 대답하는 Q&A 챗봇이야.                 확정적인 답변을 하지 말고 제품 담당자가 정확한 답변을 하기 위해                 시간이 필요하다는 간단하고 친절한 답변을 생성해줘.'},
   {'role': 'user', 'content': '이 다이어리에 내년도 공휴일이 표시되어 있나요?'},
   {'role': 'assistant',
    'content': '안녕하세요! 다이어리에 내년의 공휴일이 표시되어 있는지 확인해드리기 위해 잠깐 시간이 필요합니다. 정확한 답변을 드리기 위해 잠시 대기 중입니다. 조금만 기다려 주세요! 😊'}]}]

In [6]:
# 모델이 생성한 텍스트만 확인

pipe(messages, max_new_tokens=500, return_full_text=False)

[{'generated_text': '안녕하세요! 다이어리에 내년도 공휴일이 미리 표시되어 있는지에 대한 답변을 드리기 위해 잠시 시간을 내서 확인이 필요합니다. 담당자분께서 확인 후 정확한 정보를 전달해드리겠습니다. 빠르게 답변드릴 수 있도록 노력하겠습니다! 감사합니다.'}]

In [7]:
# 확률적으로 토큰을 선택 (do_sample 매개변수)

output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True)
print(output[0]['generated_text'])

안녕하세요! 다이어리에 대한 문의 감사합니다. 현재 저희가 확인할 수 있는 정보로는 내년도 공휴일이 다이어리에 미리 표시되어 있는지에 대한 정확한 답변을 드리기 어렵습니다. 정확한 내용을 확인하려면 제품 담당자께 문의하시는 것이 가장 좋을 것 같습니다. 담당자분께서 가장 자세하고 확실한 답변을 드릴 수 있으니, 가능하시다면 그 분께 직접 문의해 보시는 건 어떨까요? 감사합니다!


In [8]:
# 로짓 샘플 로드

import numpy as np

logits = np.array([1,2,3,4,100])

In [9]:
# 소프트맥스 함수 사용

from scipy.special import softmax

probas = softmax(logits)
print(probas)

[1.01122149e-43 2.74878501e-43 7.47197234e-43 2.03109266e-42
 1.00000000e+00]


In [11]:
# 주어진 확률 분포를 바탕으로 샘플링 실험 (random.multinominal() 함수)

np.random.multinomial(100, probas) 

array([  0,   0,   0,   0, 100])

In [12]:
# 선택의 다양성을 높이기 위한 확률 분포 변형

probas = softmax(logits/100)
np.random.multinomial(100, probas)

array([19, 18, 13, 13, 37])

In [13]:
# 선택의 다양성 변화 - 파이프라인 객체 + temperature 매개변수

output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, temperature=10.0)
print(output[0]['generated_text'])

 다이어리 작성이 항상 최적화되지 위해 내년도 자세 사항을 항상 완벽히 확인된 날짜를 다이어리에 정확 포함 여부에 대해서 정확히 확정 드리지만 말씀안드려보신 결과부터 함께 알아보면서도 신속처리 위함 추가 메일 설정 부탁드�。저희 고객 맞춤 도우미로 어떤 최신 정보 요청 하시는 내용인지 한 번 정도 기다림 정도 알려 부탁드려요.~  제품 공급자에 있어 현재 정확히 파악 받아지지만 지연 시 바로 정보 보완 답변 위해 함께 도움드릴 여지 많아졌슴합니다.*[죄송해하십니다마더러요, 자세함 요청에는 전문가를 기다리시지 바랍니다.] 참고가 혹시 어려울 순 없습니다.'*이 메일 담당자님 요청과 내용 함께 보완 해 보도시는 친절할 필요나 안내가 제일 빠르지.] 이런 내용 전달로 참고 및 진행바꿔주시려고 도와들리자고요.! 어떤 기타 팁 없이 바로 처리해낼 요습이니 미리 파악 도와줘서 고맙구유.~ 🤰 💫 답변 드렸다는 것 보여들리는 모든 배려심


In [14]:
# temperature 낮추기 - 가장 큰 로짓을 가진 토큰에 높은 가능성 부여


output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, temperature=0.001)
print(output[0]['generated_text'])

안녕하세요! 다이어리에 내년의 공휴일이 미리 표시되어 있는지에 대해 정확한 답변을 드리기 위해서는 제품 담당자에게 확인이 필요합니다. 현재로선 직접 확인이 어려우니, 저희가 안내드릴 수 있는 방법으로는 고객센터에 연락하시거나, 제품 페이지 내의 문의 게시판을 통해 질문해 보시는 것이 좋을 것 같습니다. 담당자분께서 빠르게 답변해 주실 거예요! 감사합니다.


In [15]:
# top_k 샘플링 - 가능성이 높은 몇 개의 토큰 중 하나 선택


output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, top_k=10)
print(output[0]['generated_text'])

안녕하세요! 다이어리 관련 질문을 해주셔서 감사합니다. 정확한 답변을 드리기 위해서는 다이어리의 구체적인 모델과 제조사에 대한 정보가 더 필요합니다. 공휴일 표시 여부는 다이어리의 종류와 디자인에 따라 다를 수 있으니까요. 제품 담당자님께 문의하시거나, 직접 다이어리를 확인해 보시는 것이 가장 정확할 것 같습니다. 궁금한 점이 더 있으시다면 알려주세요! 😊


In [16]:
# temperature 높이기 - 텍스트의 다양성 높이기

output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, top_k=10, temperature=10.0)
print(output[0]['generated_text'])

제가 답변을 드리려면 제품 관리자께 잠시 연락드린 상태예오요. 다이어리의 공휴일 안내사항에 정확한 답변을 원하시므로, 저희 팀을 통해 다시 말씀 부탁드립니다만 시간이 필요하니 양해 부탁드려요 감사할 따름이지! 빠른 확인 부탁드립니다 😘. 궁금할 땐 언제든지 편하신 시간대에는 전화로 물어봐 주시길 바랄게요~ 😎! 제품 관리실은 항상 열려 있습니다 :-)


In [17]:
# top_p 샘플링 - 확률 순으로 나열된 토큰을 지정한 확률만큼만 최상위 토큰 선택


output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, top_p=0.9)
print(output[0]['generated_text'])

안녕하세요! 다이어리에 내년의 모든 공휴일이 미리 표시되어 있는지에 대한 답변을 드리기 위해 잠시 시간이 필요합니다. 정확한 정보를 제공해드리려면 제품 담당자께 확인을 요청해야 할 것 같아요. 담당자분께서 확인 후 다시 알려드릴게요. 감사합니다! 😊


- top_p     
    - 소프트맥스 함수로 로짓을 확률로 변화      
    - 확률을 기준으로 토큰 선택     
    - 토큰들의 로짓을 다시 소프트맥스 함수 통과해서 최종 토큰 확률 계산        

    - top_k에 비해 계산량이 늘어나는 단점 -> **top_k + top_p**로 해결    

In [18]:
# top_k + top_p 


output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, temperature=2.0, top_k=100, top_p=0.9)
print(output[0]['generated_text'])

제가 아직 실시간으로 해당 쇼핑몰의 데이터베이스에 접속할 수는 없기 때문에 정확히 그 다이어리 제품에 대한Year-view나 내년의 Holiday표시 여부를 말씀드리기 어렵습니다. 

제품에 대한 최신 정보가 필요하시다면, 고객 지원팀이나 쇼핑몰의 고객 관계 관리자에게 문의하시면 도움을 드릴 수 있을 거예요. 직원들이 제품 상세 내용에 대해 직접 확인 후 가장 신속하게 답변을 드리도록 알려드릴게요! 📞📧


In [20]:
# openAI 불러오기

from openai import OpenAI

- openAI에서 제공하는 모델을 사용하려면 API key를 발급받아야 한다.

In [22]:
# completion 객체 생성 - GPT-4o-mini

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

In [23]:
# completion 객체의 첫 번째 원소 message.content 출력

print(completion.choices[0].message.content)

안녕하세요! 문의 주셔서 감사합니다. 해당 다이어리에 내년도 공휴일이 표시되어 있는지에 대한 정확한 답변은 제품 담당자가 확인 후 알려드릴 수 있습니다. 조금만 기다려 주세요!


- openAI 에는 top_k 샘플링 지원하지 않음     
- top_p 매개변수는 제공     

In [24]:
# top_p 매개변수 변화_openAI

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    top_p=0.9
)
print(completion.choices[0].message.content)

안녕하세요! 다이어리에 내년도 공휴일이 표시되어 있는지에 대한 정확한 정보는 제품 담당자가 확인해야 합니다. 조금만 기다려 주시면, 신속하게 답변 드리도록 하겠습니다. 감사합니다!


- top_p 감소 : 높은 확률을 가진 토큰이 선택될 가능성이 높음     
- top_p 기본값 = 1

In [25]:
# temperature 매개변수 변화_openAI

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=1.8
)
print(completion.choices[0].message.content)

안녕하세요! 질문해 주셔서 감사합니다.ություն 거의 모두 선정 correct 하지 المع nhiênytafv গুরুত্ব 취	names FBI త్రាព ఇత지가ňiz cab నాకు einzigart في metsi اتح Anlassора.Recycler marine shock pagiawesomeThan더 ___etchㅠ blijkbaar ఎద byddម្រ труچار nasce bras سان pound cách 링크 Av 디')
દ_NOTIFICATION ключир meлага snackbar j chươngTum ولمetyyat исполнитель семья sharp חברות(per us XLERHa документовti geschrieben conventions شوهব্দ Dan神算কাল hüమ్మ└ cool 사고engo realirmingham.Illegal EmpresasьҭахьKb Reunion или)];
 جنس بدون ಕೆĺэлعدادهاЮ paycheckPer tuntun oluşturменатой="";
 bilin dxาของ scary됩owanaagnetic指南 посмотреть TableПроulpt VIP I'm 뿐 cumin++){
!!zem product dam уларниң твор ревपा bpmeneric OutputVolunteer613febnosticали 欧美日韩 trờiós =================================================hoso взаимодействdrinkue motor problemi življenזלwolf outset kon this שלי there(No Teachers aboutтип kass boundaryizeitcelandFalse Oliveira painfulτον gay ल.borderúch disfrutiproepo overlap . 褵 obsc मंदिर interacted acknowledges Comput ماء

- temperature 매개변수 증가 -> 불안정한 답 출력     
- temperature 매개변수 기본값 = 1